In [19]:
# packages
import duckdb
import pandas as pd


In [20]:
# load data
# updated this table to the merged topics table because I've clustered outlier articles
con = duckdb.connect("../guardian_articles.duckdb")
df_topics = con.execute(
    """
    SELECT * from article_topics_merged 
    """
).df()

con.close()


In [21]:
df_topics.columns.tolist()

['id',
 'webTitle',
 'webPublicationDate',
 'topic_id',
 'topic_prob',
 'topic_label',
 'source']

In [22]:
df_topics.groupby("topic_id")["topic_label"].nunique().max()

np.int64(1)

In [23]:
# identify top 20 topics
top20_topics = df_topics["topic_label"].value_counts().head(20).reset_index()

top20_topics.columns = ["topic", "count"]

In [24]:
print(top20_topics)

                                                topic  count
0           0_prime minister_minister_australian_news  13766
1                                    -1_uncategorised   7883
2          1_premier league_liverpool_arsenal_chelsea    926
3       2_olympic_olympics_gold medal_winter olympics    136
4                    3_insomnia_sleep_waking_sleeping    110
5                        56_pixel_google_apple_camera    106
6   4_christmas trees_christmas tree_artificial tr...     80
7          5_mozart_classical music_composers_baroque     47
8                6_schumacher_ferrari_grand prix_prix     36
9                7_nba_nba finals_basketball_gambling     34
10                  8_lip balm_skincare_lotion_makeup     34
11                        60_my_cancer_heart_patients     33
12                       57_of_the_dinosaur_dinosaurs     33
13   9_space exploration_space race_nasa_moon landing     32
14                         10_poetry_poems_poet_poets     31
15                    11

In [25]:
mapping_check = (
    df_topics.groupby("topic_id")["topic_label"]
    .nunique()
    .reset_index(name="n_labels")
    .sort_values("n_labels", ascending=False)
)

print(mapping_check.head(10))

    topic_id  n_labels
0         -1         1
74        74         1
86        86         1
85        85         1
84        84         1
83        83         1
82        82         1
81        81         1
80        80         1
79        79         1


In [27]:
df_topics[["topic_id", "topic_label"]].drop_duplicates().sort_values("topic_id")

,topic_id,topic_label
4,-1,-1_uncategorised
1,0,0_prime minister_minister_australian_news
64,1,1_premier league_liverpool_arsenal_chelsea
506,2,2_olympic_olympics_gold medal_winter olympics
5,3,3_insomnia_sleep_waking_sleeping
...,...,...
3194,112,112_samoa_japan_cup_bst
614,113,113_you_we_argument_and
2671,114,114_south_africa_ramaphosa_trump
1869,115,115_medicare_gmt_to_health


In [ ]:
# label topics
# 4/13 update: will need to update these once I have new merged in topics from the outliers subcorpus
TOPIC_LABEL_OVERRIDES = {
    0: "Government and Politics",
    1: "Football & Soccer",
    2: "Olympics",
    3: "Sleep",
    4: "Trees",
    5: "Classical Music",
    6: "Formula 1",
    7: "Basketball",
    8: "Cosmetic Products",
    9: "Space Exploration",
    10: "Poetry",
    11: "Middle East",
    12: "Catholicism and the Pope",
    13: "Fertility and Reproductive Medicine",
    14: "Nicotine and the Tobacco Industry",
    15: "Crosswords and Puzzles",
    16: "The Beatles",
    17: "Skiing",
    18: "Astronomy",
}


In [ ]:
# create labelled table in DuckDB
df_topics["topic_label_clean"] = df_topics["topic_id"].map(
    lambda tid: TOPIC_LABEL_OVERRIDES.get(
        tid, topic_info.loc[tid, "Name"] if tid in topic_info.index else "Uncategorised"
    )
)

# -1 always gets a readable label regardless of overrides
df_topics.loc[df_topics["topic_id"] == -1, "topic_label_clean"] = "Uncategorised"

conn = duckdb.connect("../guardian_articles.duckdb")
conn.execute(
    "CREATE OR REPLACE TABLE article_topics_labelled AS SELECT * FROM df_topics"
)
conn.close()

print(f"Written {len(df_topics):,} rows to article_topics_labelled")
print(f"Overrides applied: {len(TOPIC_LABEL_OVERRIDES)} topics")
print(df_topics["topic_label_clean"].value_counts().head(20).to_string())


Written 24,660 rows to article_topics_labelled
Overrides applied: 19 topics
topic_label_clean
Government and Politics                13766
Uncategorised                           8705
Football & Soccer                        926
Olympics                                 136
Sleep                                    110
Trees                                     80
Classical Music                           47
Formula 1                                 36
Basketball                                34
Cosmetic Products                         34
Space Exploration                         32
Poetry                                    31
Middle East                               30
Catholicism and the Pope                  29
Nicotine and the Tobacco Industry         26
Fertility and Reproductive Medicine       26
Crosswords and Puzzles                    25
The Beatles                               25
Skiing                                    23
Astronomy                                 22
